# 11 — the CMS frame: both drawers open on the first click

One notebook for the frame, `packages/gatsby-plugin-jaen/src/components/JaenFrame/**`.

**What it answers.** The owner, 2026-09-08, on the live CMS: "nach dem
Bearbeiten lassen sich das Hamburger-Menue nicht mehr oeffnen, also beide
Drawer. Erst nach vielen Versuchen gehen sie irgendwann wieder auf." The cause
was measured in `docs/architecture/editing-performance.md`, "The drawer that
would not open", and it is neither of the two hypotheses that were put to it.
While a drawer stands the page is inert: the body carries
`pointer-events: none`, `#___gatsby` carries `aria-hidden`, and each 320px panel
covers its own trigger, so 56 of 56 gestures aimed at a drawer button produced
no pointer event on the button at all. They were spent dismissing the drawer
that stood, and the drawer the person asked for opened only on the next click.

**What is being verified.** The fix, on a local production build of booklimo.at
served under its own name, signed in as the booklimo human admin, against the
live agent, with edit mode on and after a field was typed. Six things:

1. the trigger is inside its own `Drawer.Root`, which is what makes it a
   trigger and not furniture beside one,
2. from a page with nothing open, the drawer opens on the first click and stays
   open, ten times out of ten, at 1440 and at 390, for both drawers,
3. one gesture goes from one drawer to the other, where it used to take two,
4. the drawer is closed by the button that opened it, which the panel covers
   and which never worked on the left at all (0 of 26 in the measurement),
5. an impatient burst leaves the button working on the very next click, and
   every one of its clicks is seen rather than swallowed,
6. all of the above with the storm reproduced on purpose, thousands of actions
   dispatched into jaen's own store in a loop from the page, and the drawer
   still standing after five seconds of it.

**What it does not do.** It changes no behaviour and publishes nothing. It types
into one field, because that is the state the owner reported the failure in, and
it sets that field back and reads it back before it ends. booklimo only, which
is where this estate tests (`okf/decisions/hard-rules.md`). Nothing is written
on limosen.

**The storm is not this notebook's to fix.** With edit mode on and nobody
touching anything the main thread is fully occupied and the frame commits about
57 times a second, which is the register loop in `packages/jaen/src/hooks/use-field.ts`
and belongs to `editing-performance.md`. It is measured here as the condition
the drawers have to work under, and the run reports it so a later reading can
tell whether it changed.

In [1]:
import json, os, pathlib, sys, time

import jaen_testkit as k

k.start_run('11-cms-frame')

REPO = pathlib.Path(k.CONFIG['repo_root'])
FRAME = REPO / 'packages' / 'gatsby-plugin-jaen' / 'src' / 'components' / 'JaenFrame'
SITE = pathlib.Path(os.environ.get('JAEN_SITE_BUILD', '/home/snekmin/git/limosen-v3/booklimo.at'))
OUT = REPO / 'tests' / 'frame'
OUT.mkdir(parents=True, exist_ok=True)
RUN_FILE = OUT / os.environ.get('JAEN_FRAME_RUN', 'run.json')
# Reuse a stored run rather than driving the browser again. Off by default: a
# notebook that reads yesterday's numbers is not a measurement.
REUSE = os.environ.get('JAEN_FRAME_REUSE', '') == '1'
PLAYWRIGHT_PYTHON = os.environ.get('JAEN_PLAYWRIGHT_PYTHON', sys.executable)

print('repo', REPO)
print('site', SITE, 'built' if (SITE / 'public' / 'index.html').is_file() else 'NOT BUILT')
print('run  ', RUN_FILE, 'reuse' if REUSE else 'drive the browser')

repo /home/snekmin/git/limosen-v3/jaen
site /home/snekmin/git/limosen-v3/booklimo.at built
run   /home/snekmin/git/limosen-v3/jaen/tests/frame/run.json drive the browser


## The frame's own source

Three readings of the code, so that a green browser run cannot be green for the
wrong reason. They are cheap and they say what the fix is: the open state is not
`useDisclosure` any more, the trigger is inside the root, and the router that
takes a gesture while the page is inert exists and is mounted once by the frame.

In [2]:
LEFT_SRC = (FRAME / 'components' / 'DrawerLeft' / 'DrawerLeft.tsx').read_text()
RIGHT_SRC = (FRAME / 'components' / 'DrawerRight' / 'DrawerRight.tsx').read_text()
STATE_SRC = (FRAME / 'drawer-state.ts').read_text() if (FRAME / 'drawer-state.ts').is_file() else ''
FRAME_SRC = (FRAME / 'JaenFrame.tsx').read_text()

with k.section('the source'):
    with k.check('the open state is outside React, in one place both drawers read') as c:
        if not STATE_SRC:
            c.fail('there is no drawer-state.ts', abort=True)
        c.expect_contains(STATE_SRC, 'useSyncExternalStore',
                          'the store is read through useSyncExternalStore')
        c.expect_true('let openDrawer' in STATE_SRC,
                      'the state is module scope, so a remount cannot reach it')
        for name, source in (('DrawerLeft', LEFT_SRC), ('DrawerRight', RIGHT_SRC)):
            # The call and not the word: DrawerLeft's own comment names
            # useDisclosure to say what it replaced.
            c.expect_not_contains(source, 'useDisclosure(',
                                  '%s keeps no state of its own' % name)
            c.expect_contains(source, "useJaenFrameDrawer('%s')" % ('left' if name.endswith('Left') else 'right'),
                              '%s reads the shared state' % name)

    with k.check("each trigger is inside its own Drawer.Root") as c:
        for name, source in (('DrawerLeft', LEFT_SRC), ('DrawerRight', RIGHT_SRC)):
            root = source.index('<Drawer.Root')
            trigger = source.index('<Drawer.Trigger')
            c.expect_true(root < trigger, '%s: the trigger is inside the root' % name)
            c.expect_contains(source, 'useJaenFrameDrawerTrigger',
                              '%s registers its trigger with the router' % name)

    with k.check('the frame mounts the gesture router once') as c:
        c.expect_contains(FRAME_SRC, 'useJaenFrameDrawerRouter()', 'mounted by JaenFrame')
        c.expect_contains(STATE_SRC, "window.addEventListener('pointerdown', onPointerDown, true)",
                          'a capture listener on window, which runs before the document')
        c.expect_contains(STATE_SRC, 'SETTLE_MS', 'the opening animation has a settle window')

## The run

`tests/support/frame-drawer.py`. It serves booklimo's own production build
behind a socat TLS listener under `https://booklimo.at`, because the OIDC
client, the agent and the storage gateway all check the origin, signs in as the
booklimo human admin, turns edit mode on through an init script (writing the
persisted store and reloading does not work: the running page persists its own
`isEditing: false` back within a second), types into one field, and drives every
gesture as `mouse.move`, `mouse.down`, `mouse.up` at the button's own
coordinates. Never `locator.click()`, which re-resolves and retries and would
hide the very failure this measures.

It takes about nine minutes.

In [3]:
RUN = None

with k.section('the run'):
    with k.check('the verifier drove the browser') as c:
        if REUSE and RUN_FILE.is_file():
            RUN = json.loads(RUN_FILE.read_text())
            c.warn('reused %s, no browser was driven' % RUN_FILE.name)
        else:
            if not (SITE / 'public' / 'index.html').is_file():
                c.skip('booklimo has no production build in %s' % SITE)
            r = c.require(k.sh('%s tests/support/frame-drawer.py' % PLAYWRIGHT_PYTHON,
                               cwd=str(REPO), timeout=1200, label='frame-drawer'),
                          'the verifier')
            RUN = r.json()
            if RUN is None:
                c.fail('the verifier printed no JSON: %s' % r.evidence(200), abort=True)
            RUN_FILE.write_text(json.dumps(RUN, indent=1))
        c.expect_true(RUN.get('signedIn'), 'signed in as the booklimo human admin')
        c.expect_equal(RUN.get('error'), None, 'the run reached the end')
        c.note('stored in tests/frame/%s' % RUN_FILE.name)

    with k.check('edit mode is on and the page is the CMS') as c:
        editing = (RUN or {}).get('editing') or {}
        c.expect_true(editing.get('entered'), 'edit mode entered')
        c.expect_true((editing.get('editableFields') or 0) > 10,
                      '%s editable fields on the page' % editing.get('editableFields'))

    with k.check('the storm is present, which is the condition the drawers work under') as c:
        idle = ((RUN or {}).get('editing') or {}).get('idle') or {}
        c.note('%s commits/s, %s frames/s, %s localStorage writes/s with nobody typing'
               % (idle.get('commitsPerSecond'), idle.get('framesPerSecond'),
                  idle.get('writesPerSecond')))
        # Not a gate on this notebook. It is a WARN so that the day the storm is
        # fixed this reading changes visibly rather than silently.
        if (idle.get('commitsPerSecond') or 0) > 10:
            c.warn('the register storm of editing-performance.md is still running')
        else:
            c.ok('the storm is gone, which is the field path\'s work and not the frame\'s')

print(json.dumps({'signedIn': (RUN or {}).get('signedIn'),
                  'trigger': (RUN or {}).get('trigger'),
                  'editing': (RUN or {}).get('editing')}, indent=1))

{
 "signedIn": true,
 "trigger": {
  "left": {
   "tag": "button",
   "ariaHasPopup": "dialog",
   "ariaExpanded": "false",
   "ariaControls": null,
   "dataScope": "dialog",
   "dataPart": "trigger",
   "box": {
    "x": 16,
    "y": 14,
    "w": 36,
    "h": 36
   }
  },
  "right": {
   "tag": "button",
   "ariaHasPopup": "dialog",
   "ariaExpanded": "false",
   "ariaControls": null,
   "dataScope": "dialog",
   "dataPart": "trigger",
   "box": {
    "x": 1388,
    "y": 14,
    "w": 36,
    "h": 36
   }
  }
 },
 "editing": {
  "entered": true,
  "editableFields": 41,
  "idle": {
   "seconds": 5,
   "commitsPerSecond": 58.4,
   "framesPerSecond": 29.4,
   "writesPerSecond": 3.2,
   "mounts": {
    "left": 0,
    "right": 0
   },
   "inserted": {
    "left": 0,
    "right": 0
   }
  }
 }
}


## The trigger is the drawer's own

Read off the live DOM rather than off the source: Ark stamps `data-scope="dialog"`
and `data-part="trigger"` on a button that is inside its root, and gives it
`aria-haspopup="dialog"` and `aria-expanded`. A button beside the root carries
none of them, which is what the two drawers had.

`aria-controls` is null while the drawer is shut, because Ark names the panel
only while there is a panel to name. That is also what the run uses to decide
which drawer is open, and it is why it can: geometry cannot. At 390 the right
panel is 320px wide on a 390px screen, so it stands at x 70, in the left half of
the viewport, and it crosses the middle while it slides. The first cut of this
run classified by x and reported the right drawer as never opening at 390 when
it had opened every time.

In [4]:
with k.section('the trigger'):
    trigger = (RUN or {}).get('trigger') or {}
    for side in ('left', 'right'):
        with k.check('the %s trigger is inside its own drawer' % side) as c:
            facts = trigger.get(side)
            if not facts:
                c.fail('no trigger was found in the header', abort=True)
            c.expect_equal(facts.get('dataScope'), 'dialog', 'data-scope')
            c.expect_equal(facts.get('dataPart'), 'trigger', 'data-part')
            c.expect_equal(facts.get('ariaHasPopup'), 'dialog', 'aria-haspopup')
            c.expect_true(facts.get('ariaExpanded') in ('true', 'false'),
                          'aria-expanded is %s' % facts.get('ariaExpanded'))
            c.note('at x %s y %s, %sx%s' % (facts['box']['x'], facts['box']['y'],
                                            facts['box']['w'], facts['box']['h']))

## The first click, at 1440 and at 390

From a page with nothing open, ten rounds per drawer per width. Every round
shuts everything first, clicks once, and reads the drawer's own panel by the id
its trigger names, sampled every animation frame so a drawer that opens and is
thrown away milliseconds later is seen rather than missed.

Two numbers have to be ten: the drawer opened, and it was still open a second
and a half later.

In [5]:
with k.section('the first click'):
    for width, widths in ((w, v) for w, v in sorted(((RUN or {}).get('firstClick') or {}).items())):
        for side, result in sorted(widths.items()):
            with k.check('%s: the %s drawer opens on the first click and stays open'
                         % (width, side)) as c:
                c.expect_equal(result['startedShut'], result['rounds'],
                               'every round started with nothing open')
                c.expect_equal(result['openedOnFirstClick'], result['rounds'],
                               '%d of %d opened on the first click'
                               % (result['openedOnFirstClick'], result['rounds']))
                c.expect_equal(result['stayedOpen'], result['rounds'],
                               '%d of %d still open afterwards'
                               % (result['stayedOpen'], result['rounds']))
                ms = result['openedAtMs']
                if ms:
                    c.note('open at %s to %s ms, median %s' % (ms[0], ms[-1], ms[len(ms) // 2]))

    with k.check('the drawers are never remounted, at either width') as c:
        mounts = 0
        rounds = 0
        for widths in ((RUN or {}).get('firstClick') or {}).values():
            for result in widths.values():
                rounds += result['rounds']
                for attempt in result['attempts']:
                    mounts += attempt['mounts']['left'] + attempt['mounts']['right']
                    mounts += attempt['inserted']['left'] + attempt['inserted']['right']
        c.expect_equal(mounts, 0, '0 remounts and 0 insertions over %d gestures' % rounds)

## From one drawer to the other, and the button that opened it

The two gestures the measurement found impossible.

**One drawer to the other.** With a drawer standing, a click on the other
drawer's button used to reach nothing: it dismissed what stood, and the drawer
that was asked for came on the next click. It is one click now, and the page is
still inert while it happens, which is the point. The modality is not removed,
the frame takes the gesture in front of it: the `window` listener installed
before any script of the page sees the `pointerdown` and the `document` listener
does not, which is the router doing its work.

**The button that opened it.** The left drawer could not be closed by its own
button at all, 0 of 26, because its panel covers the trigger and the point lands
on the drawer's own header. It closes now, and the reading of what is under the
pointer says why the old code could not: it is the drawer, not the button.

In [6]:
with k.section('one drawer to the other'):
    cross = (RUN or {}).get('cross') or {}
    with k.check('one gesture goes from one drawer to the other') as c:
        c.expect_equal(cross.get('switchedOnOneClick'), cross.get('rounds'),
                       '%s of %s' % (cross.get('switchedOnOneClick'), cross.get('rounds')))
        for pair in cross.get('pairs', []):
            c.expect_true(pair['secondOpened'] and not pair['firstStillOpen'],
                          '%s to %s' % (pair['from'], pair['to']))

    with k.check('the page is still inert while a drawer stands, and the frame takes the gesture') as c:
        for pair in cross.get('pairs', []):
            inert = pair.get('inertBefore') or {}
            c.expect_equal(inert.get('bodyPointerEvents'), 'none', 'body pointer-events')
            c.expect_equal(inert.get('rootAriaHidden'), 'true', '#___gatsby aria-hidden')
            c.expect_true(pair['windowSawPointerdown'], 'the window listener saw the gesture')
            c.expect_true(not pair['documentSawPointerdown'],
                          'the document listener did not, so the router stopped it')

with k.section('the button that opened it'):
    for side, result in sorted(((RUN or {}).get('ownTrigger') or {}).items()):
        with k.check('the %s drawer closes on its own button' % side) as c:
            c.expect_equal(result['closedByItsOwnButton'], result['rounds'],
                           '%d of %d' % (result['closedByItsOwnButton'], result['rounds']))
            under = sorted({a['topAtTrigger'] for a in result['attempts']})
            c.note('under the pointer at the trigger while open: %s' % ', '.join(map(str, under)))

## An impatient burst

What is not asserted here is the parity of the burst. The trigger is a toggle
and a toggle toggles: `aria-expanded` says so on the button, and a burst that
runs past the settle window of `drawer-state.ts` opens and then closes, which is
what any menu button does. The nominal gap is not the gap either, because every
playwright mouse call is a round trip to a main thread the storm keeps busy: four
clicks 100 ms apart really span about half a second.

What must be true is the owner's sentence turned around. Before, "a burst is one
click seen", because the dismissable layer is not armed inside the opening
animation. Now every click of the burst is seen, and the button works on the
very next one rather than after many attempts.

In [7]:
with k.section('a burst'):
    for side, result in sorted(((RUN or {}).get('burst') or {}).items()):
        with k.check('a burst on the %s button leaves it working' % side) as c:
            c.expect_equal(result['nextClickOpened'], result['rounds'],
                           'the next click opened it %d of %d times'
                           % (result['nextClickOpened'], result['rounds']))
            for attempt in result['attempts']:
                c.expect_equal(attempt['pointerdownsSeen'], attempt['clicks'],
                               'every click of the burst was seen')
            spans = [a['realSpanMs'] for a in result['attempts']]
            c.note('%d clicks a nominal %d ms apart really span %s ms'
                   % (result['attempts'][0]['clicks'], result['attempts'][0]['gapMs'], spans))
            c.note('open at the end of the burst: %d of %d, which is the toggle'
                   % (result['openAtEnd'], result['rounds']))

## The storm, reproduced on purpose

A frame that re-renders is a normal thing and a drawer must not lose its state
to one, so the storm is made rather than waited for. The store is jaen's own
module singleton and is on no global, so the run takes it off react-redux's
Provider fiber through the devtools hook it installed before React booted, and
dispatches `pages/field_register` in a `requestAnimationFrame` loop.

That action and no other, for two reasons. It is the action the register loop
in `use-field.ts` actually dispatches, so this is the storm and not a storm. And
it is one the recorder in `remote-state.ts` does not translate into a change, so
a loop of thousands reaches no agent and no draft. The run proves that rather
than claiming it: the outbox and the revision are read before and after.

In [8]:
storm = (RUN or {}).get('storm') or {}

with k.section('the storm'):
    with k.check('the storm was dispatched into jaen\'s own store') as c:
        start = storm.get('start') or {}
        if start.get('error'):
            c.fail('the store was not reached: %s' % json.dumps(start)[:200], abort=True)
        stop = storm.get('stop') or {}
        c.expect_true(start.get('started'), 'the loop started')
        c.expect_true((stop.get('dispatched') or 0) > 1000,
                      '%s actions dispatched over %.0f s'
                      % (stop.get('dispatched'), stop.get('seconds') or 0))
        cost = storm.get('cost') or {}
        c.note('under it: %s commits/s, %s frames/s' % (cost.get('commitsPerSecond'),
                                                        cost.get('framesPerSecond')))

    with k.check('nothing of the storm reached the agent') as c:
        before = storm.get('outboxBefore') or {}
        after = storm.get('outboxAfter') or {}
        c.expect_equal(after.get('outbox'), before.get('outbox'),
                       'the outbox is %s before and after' % before.get('outbox'))
        c.expect_equal(after.get('revision'), before.get('revision'),
                       'the draft revision is %s before and after' % before.get('revision'))

    for side, result in sorted((storm.get('firstClick') or {}).items()):
        with k.check('under the storm the %s drawer opens on the first click' % side) as c:
            c.expect_equal(result['openedOnFirstClick'], result['rounds'],
                           '%d of %d' % (result['openedOnFirstClick'], result['rounds']))
            c.expect_equal(result['stayedOpen'], result['rounds'], 'and stayed open')

    with k.check('a standing drawer survives the storm') as c:
        survives = storm.get('survives') or {}
        c.expect_true(survives.get('opened'), 'it opened')
        c.expect_true(survives.get('openAfterFiveSeconds'),
                      'still open after five seconds of the storm')
        c.expect_true((survives.get('commits') or 0) > 100,
                      '%s React commits in those five seconds' % survives.get('commits'))
        mounts = survives.get('mounts') or {}
        inserted = survives.get('inserted') or {}
        c.expect_equal(sum(mounts.values()) + sum(inserted.values()), 0,
                       'and no remount of either trigger')

## What the run left

One field was typed into, because that is the state the owner reported, and it
is set back and read back before the run ends. Nothing is published: the draft's
`publishedRevision` is where it was, and the difference between it and
`revision` is another session's unpublished work, left standing the way it was
found.

In [9]:
with k.section('what the run left'):
    with k.check('the field this run typed into is set back') as c:
        typed = (RUN or {}).get('typed') or {}
        back = (RUN or {}).get('setBack') or {}
        if not typed.get('original'):
            c.skip('the run typed into no field')
        read_back = back.get('readBack') or []
        c.expect_true(typed['original'] in read_back,
                      'read back as %r' % typed['original'])
        c.expect_true(all(typed['original'] + ' frame probe' != value for value in read_back),
                      "the run's own suffix is gone")

    with k.check('the run published nothing') as c:
        before = ((RUN or {}).get('typed') or {}).get('after') or {}
        after = ((RUN or {}).get('setBack') or {}).get('remoteAfter') or {}
        if not after:
            c.skip('no draft state was read back')
        c.expect_equal(after.get('publishedRevision'), before.get('publishedRevision'),
                       'publishedRevision %s throughout' % after.get('publishedRevision'))
        c.expect_equal(after.get('outbox'), 0, 'the outbox is empty, so nothing is unsent')
        c.note('revision %s, publishedRevision %s at the end'
               % (after.get('revision'), after.get('publishedRevision')))

## Summary

In [10]:
k.summary('11 — the CMS frame')
k.save_results(str(REPO / 'tests' / 'results-11-cms-frame.json'))
k.verdict()

#,Status,Section,Check,Evidence
1,PASS,the source,"the open state is outside React, in one place both drawers read","the store is read through useSyncExternalStore. the state is module scope, so a remount cannot reach it. DrawerLeft keeps no state of its own. DrawerLeft reads the shared state. DrawerRight keeps no state of its own. DrawerRight reads the shared state"
2,PASS,the source,each trigger is inside its own Drawer.Root,DrawerLeft: the trigger is inside the root. DrawerLeft registers its trigger with the router. DrawerRight: the trigger is inside the root. DrawerRight registers its trigger with the router
3,PASS,the source,the frame mounts the gesture router once,"mounted by JaenFrame. a capture listener on window, which runs before the document. the opening animation has a settle window"
4,PASS,the run,the verifier drove the browser,signed in as the booklimo human admin. the run reached the end. stored in tests/frame/run.json
5,PASS,the run,edit mode is on and the page is the CMS,edit mode entered. 41 editable fields on the page
6,WARN,the run,"the storm is present, which is the condition the drawers work under","58.4 commits/s, 29.4 frames/s, 3.2 localStorage writes/s with nobody typing. the register storm of editing-performance.md is still running"
7,PASS,the trigger,the left trigger is inside its own drawer,"data-scope. data-part. aria-haspopup. aria-expanded is false. at x 16 y 14, 36x36"
8,PASS,the trigger,the right trigger is inside its own drawer,"data-scope. data-part. aria-haspopup. aria-expanded is false. at x 1388 y 14, 36x36"
9,PASS,the first click,1440x900: the left drawer opens on the first click and stays open,"every round started with nothing open. 10 of 10 opened on the first click. 10 of 10 still open afterwards. open at 0 to 6 ms, median 6"
10,PASS,the first click,1440x900: the right drawer opens on the first click and stays open,"every round started with nothing open. 10 of 10 opened on the first click. 10 of 10 still open afterwards. open at 0 to 7 ms, median 0"


0